In [1]:
import os
import shutil

# 定义输入路径 (请根据你实际在 Input 侧边栏看到的文件名修改路径)
exdark_path = '/kaggle/input/ExDark-Dataset' 
zero_dce_path = '/kaggle/input/Zero-DCE-Code'

# 定义输出路径 (Kaggle 允许你在 /kaggle/working/ 下写入文件)
output_path = '/kaggle/working/Enhanced_E'

# 创建输出文件夹
os.makedirs(output_path, exist_ok=True)
print("输出文件夹已准备就绪:", output_path)

输出文件夹已准备就绪: /kaggle/working/Enhanced_E


In [2]:
import os
import torch
import torchvision
from torchvision import transforms
from PIL import Image

# 1. 准备输出路径
output_path = '/kaggle/working/Enhanced_E'
os.makedirs(output_path, exist_ok=True)
print(f"输出文件夹已准备就绪: {output_path}")

# 2. 智能搜索 ExDark 数据集里的所有图片
print("正在地毯式搜索图片文件...")
image_list = []

# 遍历 Kaggle 的输入目录
for dirname, _, filenames in os.walk('/kaggle/input'):
    # 智能避开 Zero-DCE 代码文件夹，防止误处理代码包里自带的测试样图
    if 'zero' in dirname.lower() or 'dce' in dirname.lower():
        continue
    
    for filename in filenames:
        # 只要后缀是图片格式，通通抓起来
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_list.append(os.path.join(dirname, filename))

print(f"✅ 成功找到 {len(image_list)} 张待处理的低光照图片！")

# 3. 开始批量推理处理
if len(image_list) > 0:
    transform = transforms.Compose([transforms.ToTensor()])
    
    print("🚀 开始进行暗光增强处理 (加入了防爆显存机制，请耐心等待)...")
    
    # 推理模式，不计算梯度，省显存提速
    with torch.no_grad(): 
        for i, img_path in enumerate(image_list):
            # 获取原图所在的类别文件夹名（例如 'Bicycle'）和文件名
            folder_name = os.path.basename(os.path.dirname(img_path))
            file_name = os.path.basename(img_path)
            
            # 组合保存的文件名：类别_文件名.jpg
            save_name = f"{folder_name}_{file_name}"
            save_path = os.path.join(output_path, save_name)
            
            try:
                # 1. 读取图片并转换为 RGB
                original_img = Image.open(img_path).convert('RGB')
                
                # --- 核心新增：防爆显存机制 (限制最大分辨率) ---
                MAX_SIZE = 1200 # 设定最长边限制为 1200 像素
                if max(original_img.size) > MAX_SIZE:
                    # thumbnail 会按原图比例安全缩小，不会导致图片变形失真
                    # 兼容不同版本的 PIL，优先使用 Image.Resampling.LANCZOS，如果没有则用 Image.LANCZOS
                    resample_method = getattr(Image, 'Resampling', Image).LANCZOS
                    original_img.thumbnail((MAX_SIZE, MAX_SIZE), resample_method)
                # ----------------------------------------
                
                # 2. 预处理并送入 GPU
                img_tensor = transform(original_img).unsqueeze(0).to(device)
                
                # 3. 模型推理 (Zero-DCE 返回三个参数，中间的 enhanced_tensor 是我们要的增强图像)
                _, enhanced_tensor, _ = DCE_net(img_tensor) 
                
                # 4. 保存增强后的图片
                torchvision.utils.save_image(enhanced_tensor, save_path)
                
                # 5. 主动清理显存垃圾，防止碎片积累导致后期 OOM
                del img_tensor, enhanced_tensor
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"❌ 处理图片 {img_path} 时出错: {e}")
                # 即使出错了也强制清理一下显存，防止连环崩溃
                torch.cuda.empty_cache()
            
            # 进度条打印：每处理 100 张或者到最后一张时，打印一次进度
            if (i + 1) % 100 == 0 or (i + 1) == len(image_list):
                print(f"进度: 已处理 {i + 1} / {len(image_list)} 张图片...")

    print("🎉 所有图片增强处理完毕！你可以执行下一阶段的代码进行打包下载了。")
else:
    print("❌ 还是没有找到图片。请检查 Kaggle 右侧边栏的 'Input' 区域。")

KeyboardInterrupt: 

In [ ]:
import shutil

# 将 Enhanced_E 文件夹打包成 Enhanced_E.zip
shutil.make_archive('/kaggle/working/Enhanced_E_dataset1', 'zip', '/kaggle/working/Enhanced_E')

print("打包完成！请在右侧 Output 区域刷新并下载 Enhanced_E_dataset1.zip")